# Cross-Market Volatility Forecasting

This Colab notebook runs the comparison of daily variance forecasts for the S&P 500 and TASI using chronological tests, statistical models, and machine learning. The target is the next session's squared log return, a noisy proxy for latent volatility. The results are not investment advice.

Raw data are not included in the repository. Use price files you are licensed to use and check the provider's terms.

## 1. Clone the repository and install dependencies

After the updates are merged on GitHub, run the next cells.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/alrashedinaifrr/financial-volatility-forecasting.git'
PROJECT_DIR = Path('/content/financial-volatility-forecasting')
if not (PROJECT_DIR / 'src' / 'portfolio_v7.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'src' / 'portfolio_v7.py').exists()
print('Project:', PROJECT_DIR)

In [ ]:
%cd /content/financial-volatility-forecasting
%pip install -q -r requirements.txt

## 2. Add the price files

Upload or copy `sp500.csv` and `tasi.csv` to a Drive folder. Each file must contain `Date` and `Close` columns; the pipeline sorts the dates. Do not upload confidential data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/volatility_data')
for filename in ['sp500.csv', 'tasi.csv']:
    path = DATA_DIR / filename
    print(('✓' if path.is_file() else '✗'), path)
if not all((DATA_DIR / f).is_file() for f in ['sp500.csv','tasi.csv']):
    raise FileNotFoundError('Place sp500.csv and tasi.csv in the selected folder or update DATA_DIR.')

In [ ]:
import pandas as pd
for filename in ['sp500.csv','tasi.csv']:
    df = pd.read_csv(DATA_DIR / filename)
    missing = {'Date','Close'} - set(df.columns)
    if missing: raise ValueError(f'{filename} missing columns: {sorted(missing)}')
    print(filename, 'rows=', len(df), 'date range=', df.Date.min(), '→', df.Date.max())
    display(df[['Date','Close']].head())

## 3. Run the tests and analysis

The analysis may take some time because it performs bootstrap resampling. Each run records input SHA-256 hashes and package versions.

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q', 'tests'], cwd=PROJECT_DIR, check=True)
OUTPUT_DIR = Path('/content/volatility_results')
subprocess.run([
    'python', str(PROJECT_DIR / 'src' / 'portfolio_v7.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
], check=True)

In [ ]:
from IPython.display import display, Image
for name in ['fold_scores.csv','pairwise_qlike_comparisons.csv','aligned_transfer.csv','run_metadata.json']:
    path = OUTPUT_DIR / name
    print('\n', name)
    if path.suffix == '.csv': display(pd.read_csv(path).head(20))
    else: print(path.read_text()[:4000])
for path in sorted(OUTPUT_DIR.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/volatility_results', 'zip', OUTPUT_DIR)
print('Results archive:', archive)
files.download(archive)

## Interpretation limits

- Different trading calendars and market structures mean market scores should not be read as a direct comparison of predictability.
- Bootstrap intervals are descriptive and exploratory. Holm correction applies to the specified pairwise tests; it does not make the findings general evidence.
- This experiment does not test the profitability of a trading strategy. See `TECHNICAL_REPORT.md` and `VALIDATION_STATUS.md`.